# Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PODATKI = Path("../11_vaje/navodila")
if not PODATKI.exists():
    PODATKI = Path("../../11_vaje/navodila")
d2r = np.deg2rad

# Lokacija in Sonce (model z vaje 11)
phi, lam, S0 = 46.559, 15.639, 1361.0

# Plastenjak: dolga os v smeri sever-jug
a, b, h = 20.0, 5.0, 3.0                         # dolžina, širina, višina [m]
ploskve = {                                      # ime: (površina [m^2], naklon beta [°], azimut gama [°])
    "sever": (b * h, 90, 0), "vzhod": (a * h, 90, 90), "jug": (b * h, 90, 180),
    "zahod": (a * h, 90, 270), "streha": (a * b, 0, 0)}
S_ovoj = sum(p[0] for p in ploskve.values())      # površina ovoja
C = a * b * h * 1.2 * 1005 + a * b * 0.1 * 2.0e6  # toplotna kapaciteta: zrak + 10 cm tal [J/K]

# Funkcije

In [ ]:
# --- Lega Sonca in obsevanje (vaja 11) ---
def deklinacija(n):
    return 23.44 * np.sin(2 * np.pi * (284 + n) / 365)


def casovna_enacba(n):
    B = 2 * np.pi * (n - 81) / 364
    return 9.87 * np.sin(2 * B) - 7.53 * np.cos(B) - 1.5 * np.sin(B)


def lega_sonca(n, t_utc):
    """Višina in azimut Sonca [°] za dan n in čas t_utc [h]."""
    d = deklinacija(n)
    H = 15 * (t_utc + lam / 15 + casovna_enacba(n) / 60 - 12)
    sa = np.sin(d2r(phi)) * np.sin(d2r(d)) + np.cos(d2r(phi)) * np.cos(d2r(d)) * np.cos(d2r(H))
    alt = np.rad2deg(np.arcsin(sa))
    c = (np.sin(d2r(d)) - sa * np.sin(d2r(phi))) / (np.cos(d2r(alt)) * np.cos(d2r(phi)))
    A = np.rad2deg(np.arccos(np.clip(c, -1, 1)))
    return alt, np.where(H > 0, 360 - A, A)


def direktno(alt, n):
    """Direktno obsevanje pravokotne ploskve v jasnem vremenu [W/m^2] (0 pod obzorjem)."""
    alt = np.asarray(alt, float)
    AM = 1 / (np.sin(d2r(np.maximum(alt, 0.01))) + 0.50572 * (np.maximum(alt, 0.01) + 6.07995)**(-1.6364))
    I_D = S0 * (1 + 0.033 * np.cos(2 * np.pi * n / 365)) * 0.7**(AM**0.678)
    return np.where(alt > 0, I_D, 0.0)


# --- Obsevanje ploskev ---
def cos_vpadni(alt, A, beta, gama):
    """Kosinus vpadnega kota žarkov na ploskev z naklonom beta in azimutom gama (negativno -> 0)."""
    # TODO: cos(theta) = sin(alt) cos(beta) + cos(alt) sin(beta) cos(A - gama)
    pass


def obsevanje_ploskve(alt, A, n, beta, gama):
    """Obsevanje ploskve v jasnem vremenu [W/m^2]: direktno + razpršeno (10 % I_D sin(alt) (1 + cos beta)/2)."""
    I_D = direktno(alt, n)
    # TODO
    pass


# --- Toplotna bilanca ---
def korak_euler(T, T_zun, P_sonce, UA, dt):
    """En korak eksplicitne Eulerjeve metode za C dT/dt = P_sonce - UA (T - T_zun)."""
    # TODO
    pass


def korak_tocen(T, T_zun, P_sonce, UA, dt):
    """Točna rešitev za konstantna T_zun in P_sonce v koraku: relaksacija k T_ravn s tau = C/UA."""
    # TODO: T_ravn = T_zun + P_sonce / UA
    pass


def simulacija(vreme, U=6.0, tau_folija=0.8, T_min=None, T0=10.0, korak=korak_tocen):
    """Urna simulacija temperature v plastenjaku. Če je T_min podan, grelec ne dovoli T < T_min.
    Vrne (T [°C], moč gretja [W]) za vse ure."""
    UA = U * S_ovoj
    dt = 3600.0
    T = T0
    T_v, P_ogr = np.empty(len(vreme)), np.zeros(len(vreme))
    for k, (T_zun, P_s) in enumerate(zip(vreme["T"].to_numpy(), vreme["P_sonce"].to_numpy())):
        T = korak(T, T_zun, tau_folija * P_s, UA, dt)
        # TODO: če je T < T_min, potrebno energijo C (T_min - T) pripiši grelcu in postavi T = T_min
        pass
        T_v[k] = T
    return T_v, P_ogr

# Testno območje

In [ ]:
print(f"S_ovoj = {S_ovoj} m^2, C = {C/1e6:.2f} MJ/K, tau (U = 6) = {C / (6 * S_ovoj) / 3600:.2f} h")
# P1: poldne 21. 6.
alt, A = lega_sonca(172, 12 - lam / 15 - casovna_enacba(172) / 60)
print(f"21. 6. opoldne: alt = {float(alt):.2f}°, A = {float(A):.1f}°")
for ime, (S, beta, gama) in ploskve.items():
    print(f"  {ime:6s}: cos(theta) = {float(cos_vpadni(alt, A, beta, gama)):.3f}")

# Naloga 1 - Slika 2

In [ ]:
t = np.arange(0, 24, 1 / 6)
fig = plt.figure(figsize=(12, 4))
# TODO: obsevanje petih ploskev čez dan 21. 6. in 21. 12.; dnevna energija na m^2 (Preglednica 1)
fig.tight_layout()
plt.show()

# Naloga 2

In [ ]:
vreme = pd.read_csv(PODATKI / "Vreme_urni.csv")
cas = pd.to_datetime(vreme["date"] + " " + vreme["time"])
n_ura = cas.dt.dayofyear.to_numpy()
t_utc = cas.dt.hour.to_numpy() + 0.5 - 1          # sredina ure; čas v podatkih je CET (UTC+1)
alt_u, A_u = lega_sonca(n_ura, t_utc)
# TODO: obsevanje vodoravne ploskve (model) in dnevni indeks jasnosti K = sum(J) / sum(I_h); moč sončnega obsevanja ovoja
vreme["P_sonce"] = 0.0
mesec = cas.dt.month
tabela = pd.DataFrame({ime: [(S * obsevanje_ploskve(alt_u, A_u, n_ura, beta, gama) * vreme["K"])[mesec == m].sum() / 1000
                            for m in range(1, 13)] for ime, (S, beta, gama) in ploskve.items()}, index=range(1, 13))
print("Preglednica 2: mesečna energija na ploskve [kWh]")
print(tabela.round(0))
print("letno [MWh]:", (tabela.sum() / 1000).round(2).to_dict())

# Naloga 3 - Slika 3

In [ ]:
fig = plt.figure(figsize=(12, 4))
# TODO: temperatura brez gretja v prvih 10 dneh januarja in julija; Euler in točna rešitev (Preglednica 3)
fig.tight_layout()
plt.show()

# Naloga 4 - Slika 4

In [ ]:
T_leto, P_leto = simulacija(vreme, U=6.0, tau_folija=0.8, T_min=10.0)
fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(121)
# TODO: temperatura skozi leto z gretjem in mesečna energija gretja
fig.tight_layout()
plt.show()

In [ ]:
# Preglednica 4: letna energija za gretje (T_min = 10 °C)
# TODO